In [ ]:
# import packages

# standard library
import os
import math
import csv
import random
import json
from pathlib import Path
from dataclasses import dataclass
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
!pip install torchinfo
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau, CosineAnnealingLR
from torchinfo import summary
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download
from collections import defaultdict

# project root setup
notebook_dir = Path.cwd()
project_root = notebook_dir.parent
sys.path.insert(0, str(project_root))

# local imports

from utils.data_load import DataModule
from utils.checkpoint import ModelCheckpoint
from utils.losses import (
    # reconstruction losses
    masked_l1_loss,
    masked_huber_loss,
    masked_l1_grad_loss,
    masked_huber_grad_loss,
    masked_multires_l1_loss,
    masked_multires_l1_grad_loss,

    # diffusion losses
    diffusion_noise_mse_loss,
    diffusion_noise_l1_loss,
    diffusion_noise_huber_loss,
    masked_diffusion_noise_mse_loss,
    per_sample_mse_loss,
    per_sample_masked_mse_loss,
    min_snr,

    # latent losses
    latent_l1_loss,
    latent_l2_loss,

    # evaluation metrics
    masked_mae,
    masked_rmse,
    full_mae,
    full_rmse,
    psnr
)

In [ ]:
# set device to gpu when available
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

# load model weights from huggingface
repo_id = "han2o/inpaint_diffusion"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

root_dir = Path("/content/drive/MyDrive/music_inpainting_project/diffusion_model")
root_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# best model
# load checkpoint dictionary
best_vae = hf_hub_download(
    repo_id=repo_id,
    filename="vae/best_model.pt"
    )
# load checkpoint dictionary
vae_checkpoint = torch.load(best_vae, map_location=device)

# reinitialise model 
vae = VAE(
    in_channels=1,
    base_channels=64,
    latent_channels=8,
).to(device)

# load saved weights into model
vae.load_state_dict(vae_checkpoint["model_state_dict"])

print("Loaded best model from epoch:", vae_checkpoint["epoch"])
print("Best monitored score:", vae_checkpoint["best_score"])

In [ ]:
# freeze vae
vae.eval()
for p in vae.parameters():
    p.requires_grad=False

latent_channels = vae.latent_channels
print("Frozen VAE latent channels:", latent_channels)

In [ ]:
@torch.no_grad()
def encode2latentmean(vae, x):
    mu, logvar = vae.encode(x)
    return mu

@torch.no_grad()
def decode_from_latent(vae, z):
    return vae.decode(z)

In [ ]:
# set seed function
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

In [ ]:
# helper utilities function 

# groupnorm function due to small batch size
def make_norm(channels, max_groups=8):
    groups = min(max_groups, channels)
    while channels % groups != 0 and groups > 1:
        groups -= 1
    return nn.GroupNorm(groups, channels)

# padding as defined earlier
def padding(x, multiple=16):
    _, _, h, w = x.shape
    pad_h = (multiple - (h % multiple)) % multiple
    pad_w = (multiple - (w % multiple)) % multiple

    x_pad = F.pad(x, (0, pad_w, 0, pad_h), mode="constant", value=0.0)

    pad_info = {
        "orig_h": h,
        "orig_w": w,
        "pad_h": pad_h,
        "pad_w": pad_w,
    }
    return x_pad, pad_info

# unpadding
def unpadding(x, pad_info):
    return x[..., :pad_info["orig_h"], :pad_info["orig_w"]]

# get learning rate for scheduling and resume training
def get_lr(optimiser):
    return optimiser.param_groups[0]["lr"]

# computes fraction of the masked spectorgram for extra conditioning signal to be fed into Unet
def compute_gap_ratio(mask):
    return mask.float().mean(dim=(1, 2, 3), keepdim=True)

# frozen retrieval embedding using pooled known context latent
def pooled_context_embedding(z_context):
    return z_context.mean(dim=(2,3)) # [B, C]

##### **Diffusion Scheduler Helpers**

In [ ]:
# diffusion schedulers
@dataclass
class DiffusionSchedule:
    betas: torch.Tensor
    alphas: torch.Tensor
    alpha_bars: torch.Tensor
    sqrt_alpha_bars: torch.Tensor
    sqrt_one_minus_alpha_bars: torch.Tensor
    sqrt_recip_alphas: torch.Tensor
    posterior_variance: torch.Tensor

# cosine scheudler in OpenAI's paper
def cosine_schedule(num_steps, s=0.008, max_beta=0.999, device="cpu"):
    steps = num_steps + 1
    t = torch.linspace(0, num_steps, steps, device=device, dtype=torch.float32)
    t = t / num_steps

    f_t = torch.cos(((t + s) / (1 + s)) * math.pi * 0.5) ** 2
    alpha_bars = f_t / f_t[0]

    betas = 1.0 - (alpha_bars[1:] / alpha_bars[:-1])
    betas = torch.clamp(betas, min=1e-8, max=max_beta)

    alphas = 1.0 - betas
    alpha_bars = torch.cumprod(alphas, dim=0)

    sqrt_alpha_bars = torch.sqrt(alpha_bars)
    sqrt_one_minus_alpha_bars = torch.sqrt(1.0 - alpha_bars)
    sqrt_recip_alphas = torch.sqrt(1.0 / alphas)

    alpha_bars_prev = torch.cat(
        [torch.tensor([1.0], device=device), alpha_bars[:-1]],
        dim=0
    )
    posterior_variance = betas * (1.0 - alpha_bars_prev) / (1.0 - alpha_bars)

    return DiffusionSchedule(
        betas=betas,
        alphas=alphas,
        alpha_bars=alpha_bars,
        sqrt_alpha_bars=sqrt_alpha_bars,
        sqrt_one_minus_alpha_bars=sqrt_one_minus_alpha_bars,
        sqrt_recip_alphas=sqrt_recip_alphas,
        posterior_variance=posterior_variance,
    )

# extracts schedule values at timestep t and reshapes for broadcatsing
def extract(a, t, x_shape):
    b = t.shape[0]
    out = a.gather(0, t)
    reshape_dims = (b,) + (1,) * (len(x_shape) - 1)
    return out.view(*reshape_dims)

# forward diffusion process, adds noise to a clean latent z_0 at timestep t
def q_sample(z_0, t, noise, schedule):
    """
    samples from q(z_t | z_0) using the closed form expression:
        z_t = sqrt(alpha_bar_t) * z_0 + sqrt(1 - alpha_bar_t) * noise
    """
    sqrt_alpha_bar_t = extract(schedule.sqrt_alpha_bars, t, z_0.shape)
    sqrt_one_minus_alpha_bar_t = extract(schedule.sqrt_one_minus_alpha_bars, t, z_0.shape)
    return sqrt_alpha_bar_t * z_0 + sqrt_one_minus_alpha_bar_t * noise

# predicts clean latent z_) from a noisy latent z_t and predicted noise \episolin_hat
def predict_x0(z_t, eps_hat, t, schedule):
    """
    rearranges the forward diffusion equation to recover z_0:
        z_0 = (z_t - sqrt(1 - alpha_bar_t) * eps_hat) / sqrt(alpha_bar_t)
    """
    sqrt_alpha_bar_t = extract(schedule.sqrt_alpha_bars, t, z_t.shape)
    sqrt_one_minus_alpha_bar_t = extract(schedule.sqrt_one_minus_alpha_bars, t, z_t.shape)
    return (z_t - sqrt_one_minus_alpha_bar_t * eps_hat) / (sqrt_alpha_bar_t + 1e-8)

#### **Model Blocks**

In [ ]:
# encodes scalar diffusion timestep t into a sinusoidal embedding vector.
class SinusoidalTimeEmb(nn.Module):
    """
    same positional encoding scheme as the original transformer paper,
    but adapted for diffusion timesteps. gives the UNet a continuous,
    frequency rich representation of how noisy the input currently is.
    """
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        half_dim = self.dim // 2
        freq_factor = math.log(10000) / max(half_dim - 1, 1)
        freqs = torch.exp(
            torch.arange(half_dim, device=t.device, dtype=torch.float32) * (-freq_factor)
        )
        args = t.float().unsqueeze(1) * freqs.unsqueeze(0)
        emb = torch.cat([torch.sin(args), torch.cos(args)], dim=1)

        # pad if dim is odd
        if self.dim % 2 == 1:
            emb = F.pad(emb, (0, 1))
        return emb

# projects sinusoidal time embedding through a small MLP to increase expressiveness
class TimeEmbMLP(nn.Module):
    def __init__(self, time_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(time_dim, time_dim * 4),
            nn.SiLU(),
            nn.Linear(time_dim * 4, time_dim),
        )

    def forward(self, t_emb):
        return self.net(t_emb)

# SE block
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=4):
        super().__init__()
        hidden = max(channels // reduction, 8)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc1 = nn.Conv2d(channels, hidden, kernel_size=1)
        self.fc2 = nn.Conv2d(hidden, channels, kernel_size=1)

    def forward(self, x):
        scale = self.pool(x)
        scale = F.silu(self.fc1(scale))
        scale = torch.sigmoid(self.fc2(scale))
        return x * scale

# downsmapling
class DownSample(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv = nn.Conv2d(channels, channels, kernel_size=4, stride=2, padding=1)

    def forward(self, x):
        return self.conv(x)

# upsampling block
class UpSample(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv = nn.Conv2d(channels, channels, kernel_size=3, padding=1)

    def forward(self, x):
        x = F.interpolate(x, scale_factor=2, mode="nearest")
        return self.conv(x)

# core residual blokc for diffusion unet
class DiffusionResBlock(nn.Module):
    """
    applies two conv layers with GroupNorm and SiLU activations.

    time conditioning is injected using FiLM:
    -  the time embedding is projected to per-channel scale and shift values that modulate the intermediate feature map after the first conv.
    - SE block recalibrates channel responses after the second conv.
    - 1x1 skip connection handles channel dimension changes.
    """
    def __init__(self, in_channels, out_channels, time_emb_dim, use_se=True):
        super().__init__()

        self.norm1 = make_norm(in_channels)
        self.act1 = nn.SiLU()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)

        # projects time embedding to scale + shift for FiLM conditioning
        self.time_proj = nn.Linear(time_emb_dim, out_channels * 2)

        self.norm2 = make_norm(out_channels)
        self.act2 = nn.SiLU()
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)

        self.se = SEBlock(out_channels) if use_se else nn.Identity()
        self.skip = nn.Conv2d(in_channels, out_channels, kernel_size=1) if in_channels != out_channels else nn.Identity()

    def forward(self, x, t_emb):
        residual = self.skip(x)

        h = self.norm1(x)
        h = self.act1(h)
        h = self.conv1(h)

        film = self.time_proj(t_emb)
        scale, shift = torch.chunk(film, 2, dim=1)
        scale = scale[:, :, None, None]
        shift = shift[:, :, None, None]

        h = self.norm2(h)
        h = h * (1.0 + scale) + shift
        h = self.act2(h)
        h = self.conv2(h)
        h = self.se(h)

        return h + residual

# residual blokc with dilated convolution in the first layer
class DilatedResBlock(nn.Module):
    """
    increases the receptive field without adding parameters or reducing resolution.
    """
    def __init__(self, channels, dilation=2):
        super().__init__()
        self.norm1 = make_norm(channels)
        self.act1 = nn.SiLU()
        self.conv1 = nn.Conv2d(
            channels, channels, kernel_size=3,
            padding=dilation, dilation=dilation
        )

        self.norm2 = make_norm(channels)
        self.act2 = nn.SiLU()
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)

    def forward(self, x):
        residual = x

        h = self.norm1(x)
        h = self.act1(h)
        h = self.conv1(h)

        h = self.norm2(h)
        h = self.act2(h)
        h = self.conv2(h)

        return h + residual

# multi-head self-attention over 2D feature maps.
class SelfAttention2D(nn.Module):
    def __init__(self, channels, num_heads=4):
        super().__init__()
        assert channels % num_heads == 0

        self.channels = channels
        self.num_heads = num_heads
        self.head_dim = channels // num_heads

        self.norm = make_norm(channels)
        self.to_q = nn.Conv2d(channels, channels, kernel_size=1)
        self.to_k = nn.Conv2d(channels, channels, kernel_size=1)
        self.to_v = nn.Conv2d(channels, channels, kernel_size=1)
        self.proj = nn.Conv2d(channels, channels, kernel_size=1)

    def forward(self, x):
        b, c, h, w = x.shape
        residual = x

        x = self.norm(x)

        q = self.to_q(x).view(b, self.num_heads, self.head_dim, h * w).permute(0, 1, 3, 2)
        k = self.to_k(x).view(b, self.num_heads, self.head_dim, h * w)
        v = self.to_v(x).view(b, self.num_heads, self.head_dim, h * w).permute(0, 1, 3, 2)

        attn = torch.matmul(q, k) / math.sqrt(self.head_dim)
        attn = torch.softmax(attn, dim=-1)

        out = torch.matmul(attn, v)
        out = out.permute(0, 1, 3, 2).contiguous().view(b, c, h, w)
        out = self.proj(out)

        return out + residual

# cross-attention block where the bottleneck feature map attends to external condiitoning tokens (local context or retrieved exampels)
class CrossAttentionBlock(nn.Module):
    """
    query = spatial tokens from the bottleneck featuremap
    key/value = conditioning tokens from context or retrieval

    this allow diffusion model to be guided by additional infomraiton beynd latent and timestep
    such as surrounding spectrogram context or a retrieved reference example
    """
    def __init__(self, channels, cond_dim, num_heads=4):
        super().__init__()
        assert channels % num_heads == 0

        self.channels = channels
        self.cond_dim = cond_dim
        self.num_heads = num_heads
        self.head_dim = channels // num_heads

        self.norm = make_norm(channels)
        # queries from feature map
        self.to_q = nn.Conv2d(channels, channels, kernel_size=1)
        # keys from conditioning tokens
        self.to_k = nn.Linear(cond_dim, channels)
        # values from conditioning tokens
        self.to_v = nn.Linear(cond_dim, channels)
        self.proj = nn.Conv2d(channels, channels, kernel_size=1)

    def forward(self, x, cond_tokens):
        b, c, h, w = x.shape
        residual = x

        x_norm = self.norm(x)

        # queries and key values
        q = self.to_q(x_norm).view(b, self.num_heads, self.head_dim, h * w).permute(0, 1, 3, 2)
        k = self.to_k(cond_tokens).view(b, -1, self.num_heads, self.head_dim).permute(0, 2, 3, 1)
        v = self.to_v(cond_tokens).view(b, -1, self.num_heads, self.head_dim).permute(0, 2, 1, 3)

        attn = torch.matmul(q, k) / math.sqrt(self.head_dim)
        attn = torch.softmax(attn, dim=-1)

        out = torch.matmul(attn, v)
        out = out.permute(0, 1, 3, 2).contiguous().view(b, c, h, w)
        out = self.proj(out)

        return out + residual

#### **Conditioning Modules**

In [ ]:
# encodes local known contetx into tokens for cross attention
class ContextEncoder(nn.Module):
    """
    follows Token-Based Audio Inpainting via Discrete Diffusion

    takes the context latent and the known-region mask as input, process
    them through two conv blocks with a downsampling step, and flattens
    the spatial feature map into a sequence of tokens (B, N, cond_dim).
    
    these tokens tell the unet what the surrounding context looks like,
    helping guide reconstruction of the missing gap region.
    """
    def __init__(self, latent_channels, cond_dim=256, base_channels=64):
        super().__init__()

        in_channels = latent_channels + 1  # context latent + known-region mask

        # initial projection ot base channel dim
        self.in_conv = nn.Conv2d(in_channels, base_channels, kernel_size=3, padding=1)

        # first conv block to extract lcoal feature at ful resolution
        self.block1 = nn.Sequential(
            nn.Conv2d(base_channels, base_channels, kernel_size=3, padding=1),
            nn.GroupNorm(8, base_channels),
            nn.SiLU(),
            nn.Conv2d(base_channels, base_channels, kernel_size=3, padding=1),
            nn.GroupNorm(8, base_channels),
            nn.SiLU(),
        )

        # half spatial resolution
        self.down1 = DownSample(base_channels)

        # 2nd conv block 
        self.block2 = nn.Sequential(
            nn.Conv2d(base_channels, base_channels * 2, kernel_size=3, padding=1),
            nn.GroupNorm(8, base_channels * 2),
            nn.SiLU(),
            nn.Conv2d(base_channels * 2, base_channels * 2, kernel_size=3, padding=1),
            nn.GroupNorm(8, base_channels * 2),
            nn.SiLU(),
        )

        # project to conditioning dim expected by cross attention
        self.proj = nn.Conv2d(base_channels * 2, cond_dim, kernel_size=1)

    def forward(self, z_context, m_latent):
        known_mask = 1.0 - m_latent
        # concat latent with known region mask
        x = torch.cat([z_context, known_mask], dim=1)

        x = self.in_conv(x)
        x = self.block1(x)
        x = self.down1(x)
        x = self.block2(x)
        x = self.proj(x)

        # flatten spatila dims into token seq
        b, d, h, w = x.shape
        tokens = x.flatten(2).transpose(1, 2).contiguous()
        return tokens

# encodes retrieved context latent into tokens
class RetrievalEncoder(nn.Module):
    """
    similar architecture to ContextEncoder but without the mask channel,
    since the retrieved example is a complete latent with no missing region.

    the resulting tokens provide the unet with an external reference that
    shares acoustic structure with the gap being reconstructed.
    """
    def __init__(self, latent_channels, cond_dim=256, base_channels=64):
        super().__init__()

        self.in_conv = nn.Conv2d(latent_channels, base_channels, kernel_size=3, padding=1)

        # first conv blocks
        self.block1 = nn.Sequential(
            nn.Conv2d(base_channels, base_channels, kernel_size=3, padding=1),
            nn.GroupNorm(8, base_channels),
            nn.SiLU(),
            nn.Conv2d(base_channels, base_channels, kernel_size=3, padding=1),
            nn.GroupNorm(8, base_channels),
            nn.SiLU(),
        )

        self.down1 = DownSample(base_channels)
        
        # 2nd conv
        self.block2 = nn.Sequential(
            nn.Conv2d(base_channels, base_channels * 2, kernel_size=3, padding=1),
            nn.GroupNorm(8, base_channels * 2),
            nn.SiLU(),
            nn.Conv2d(base_channels * 2, base_channels * 2, kernel_size=3, padding=1),
            nn.GroupNorm(8, base_channels * 2),
            nn.SiLU(),
        )

        self.proj = nn.Conv2d(base_channels * 2, cond_dim, kernel_size=1)

    def forward(self, z_retrieved_context):
        x = self.in_conv(z_retrieved_context)
        x = self.block1(x)
        x = self.down1(x)
        x = self.block2(x)
        x = self.proj(x)

        b, d, h, w = x.shape
        tokens = x.flatten(2).transpose(1, 2).contiguous()  # [B,N,D]
        return tokens

# learns how to trust retrievvals
class ConfidenceGate(nn.Module):
    """
    inputs similarity score, gap ratio and normalised timsetep
    outputs a scalar of [0, 1] per sample
    """
    def __init__(self, hidden_dim=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid(),
        )

    def forward(self, sim_score, gap_ratio, t_norm):
        # all expected shape [B,1]
        x = torch.cat([sim_score, gap_ratio, t_norm], dim=1)
        return self.net(x)  # [B,1]

---

## **Diffusion U-Net**

In [ ]:
# main denoising network used by diffusion
class Diffusion_UNet(nn.Module):
    """
    the model predicts the diffusion noise that was added to z_clean.

    it also uses:
    - timestep conditioning through FiLM
    - bottleneck self-attention
    - bottleneck cross-attention over external conditioning tokens
      (local context + retrieved context)

    input channels = z_t + z_masked + mask + z0_selfcond
    cross-attends to cond_tokens at bottleneck
    """
    def __init__(self, latent_channels=8, base_channels=128, time_dim=256, cond_dim=256):
        super().__init__()

        self.latent_channels = latent_channels
        self.time_dim = time_dim
        self.cond_dim = cond_dim

        # total input channels: z_t, z_masked, mask, z0_selfcond
        in_channels = latent_channels + latent_channels + 1 + latent_channels

        # output is predicted noise in latent space
        out_channels = latent_channels

        # timestep embedding: first build sinusoidal embedding, then push through an MLP
        self.time_embed = SinusoidalTimeEmb(time_dim)
        self.time_mlp = TimeEmbMLP(time_dim)

        # initial projection from raw concatenated input into base feature width
        self.in_conv = nn.Conv2d(in_channels, base_channels, kernel_size=3, padding=1)

        # encoder 

        # first scale
        self.down1_block1 = DiffusionResBlock(base_channels, base_channels, time_emb_dim=time_dim, use_se=True)
        self.down1_block2 = DiffusionResBlock(base_channels, base_channels, time_emb_dim=time_dim, use_se=True)
        self.down1 = DownSample(base_channels)

        # second scale
        self.down2_block1 = DiffusionResBlock(base_channels, base_channels * 2, time_emb_dim=time_dim, use_se=True)
        self.down2_block2 = DiffusionResBlock(base_channels * 2, base_channels * 2, time_emb_dim=time_dim, use_se=True)
        self.down2 = DownSample(base_channels * 2)
        
        # third scale
        self.down3_block1 = DiffusionResBlock(base_channels * 2, base_channels * 4, time_emb_dim=time_dim, use_se=True)
        self.down3_block2 = DiffusionResBlock(base_channels * 4, base_channels * 4, time_emb_dim=time_dim, use_se=True)
        self.down3 = DownSample(base_channels * 4)

        # bottleneck layer

        # model combines normal conv processing, larger receptive field from dilations, self attention over latent features and cross attention over conditionin tokens
        self.mid_block1 = DiffusionResBlock(base_channels * 4, base_channels * 4, time_emb_dim=time_dim, use_se=True)
        self.mid_dilate1 = DilatedResBlock(base_channels * 4, dilation=2)
        self.mid_self_attn = SelfAttention2D(base_channels * 4, num_heads=4)
        self.mid_cross_attn = CrossAttentionBlock(base_channels * 4, cond_dim=cond_dim, num_heads=4)
        self.mid_dilate2 = DilatedResBlock(base_channels * 4, dilation=4)
        self.mid_block2 = DiffusionResBlock(base_channels * 4, base_channels * 4, time_emb_dim=time_dim, use_se=True)

        # decoder

        # first up
        self.up3 = UpSample(base_channels * 4)
        self.up3_block1 = DiffusionResBlock(base_channels * 8, base_channels * 4, time_emb_dim=time_dim, use_se=True)
        self.up3_block2 = DiffusionResBlock(base_channels * 4, base_channels * 2, time_emb_dim=time_dim, use_se=True)

        # second up
        self.up2 = UpSample(base_channels * 2)
        self.up2_block1 = DiffusionResBlock(base_channels * 4, base_channels * 2, time_emb_dim=time_dim, use_se=True)
        self.up2_block2 = DiffusionResBlock(base_channels * 2, base_channels, time_emb_dim=time_dim, use_se=True)

        # third up
        self.up1 = UpSample(base_channels)
        self.up1_block1 = DiffusionResBlock(base_channels * 2, base_channels, time_emb_dim=time_dim, use_se=True)
        self.up1_block2 = DiffusionResBlock(base_channels, base_channels, time_emb_dim=time_dim, use_se=True)
        
        # final output head mapping features back into latent noise channels
        self.out_norm = make_norm(base_channels)
        self.out_act = nn.SiLU()
        self.out_conv = nn.Conv2d(base_channels, out_channels, kernel_size=3, padding=1)
    
    # helper to resize x to match spatial shape of ref becore concatenation
    def match_spatial(self, x, ref):
        if x.shape[-2:] != ref.shape[-2:]:
            x = F.interpolate(x, size=ref.shape[-2:], mode="nearest")
        return x
    
    def forward(self, x, t, cond_tokens):
        # build timestep embedding
        t_emb = self.time_embed(t)
        t_emb = self.time_mlp(t_emb)

        x0 = self.in_conv(x) # initial projection

        # downsample path
        d1 = self.down1_block1(x0, t_emb)
        d1 = self.down1_block2(d1, t_emb)
        x1 = self.down1(d1)

        d2 = self.down2_block1(x1, t_emb)
        d2 = self.down2_block2(d2, t_emb)
        x2 = self.down2(d2)

        d3 = self.down3_block1(x2, t_emb)
        d3 = self.down3_block2(d3, t_emb)
        x3 = self.down3(d3)

        # bottleneck
        h = self.mid_block1(x3, t_emb)
        h = self.mid_dilate1(h)
        h = self.mid_self_attn(h)
        h = self.mid_cross_attn(h, cond_tokens)
        h = self.mid_dilate2(h)
        h = self.mid_block2(h, t_emb)

        # up sample with skip connections
        h = self.up3(h)
        h = self.match_spatial(h, d3)
        h = torch.cat([h, d3], dim=1)
        h = self.up3_block1(h, t_emb)
        h = self.up3_block2(h, t_emb)

        h = self.up2(h)
        h = self.match_spatial(h, d2)
        h = torch.cat([h, d2], dim=1)
        h = self.up2_block1(h, t_emb)
        h = self.up2_block2(h, t_emb)

        h = self.up1(h)
        h = self.match_spatial(h, d1)
        h = torch.cat([h, d1], dim=1)
        h = self.up1_block1(h, t_emb)
        h = self.up1_block2(h, t_emb)

        # final noise predictions
        h = self.out_norm(h)
        h = self.out_act(h)
        out = self.out_conv(h)

        return out

#### **Retrieval Bank**

In [ ]:
# retrieval bank to get top 1 retrival using frozen pooled VAE context embeddings
class RetrievalBank:
    """
    - for each training example, take the known region only
    - then encode it into a simple pooled embedding
    - save both:
      1. the pooled embedding for similarity search
      2. the full latent context for later retrieval

    later, given a query context, retrieve the most similar stored example
    """
    def __init__(self, device="cpu"):
        self.device = device
        self.bank_embeds = None   # cosine-search embeddings
        self.bank_contexts = None  # store latent context

    # build retrieval bank from a dataloader
    @torch.no_grad()
    def build(
        self,
        vae,
        dataloader,
        device,
        max_items=4000,
        desc="Building retrieval bank"
    ):
        """
        for ech sample:
            - encode masked input with frozen VAE
            - keep only the known region
            - compute a pooled embedding for similarity search
            - store the full context latent for later use
        """
        embeds = []
        contexts = []
        n_total = 0

        vae.eval()

        for batch in tqdm(dataloader, desc=desc):
            x = batch["x"].to(device, non_blocking=True)
            m = batch["mask"].to(device, non_blocking=True)
            
            # pad before encoding so shapes are valid
            x, _ = padding(x, multiple=16)
            m, _ = padding(m, multiple=16)

            # encode masked spectrogram into latent space
            z_masked = encode2latentmean(vae, x)
            m_latent = F.interpolate(m, size=z_masked.shape[-2:], mode="nearest")
            z_context = z_masked * (1.0 - m_latent)

            emb = pooled_context_embedding(z_context) # build pooled embedding use for similarity search 

            embeds.append(emb.cpu())
            contexts.append(z_context.cpu())

            n_total += x.shape[0]
            if n_total >= max_items:
                break
        
        # concatenate all stored items
        self.bank_embeds = torch.cat(embeds, dim=0)[:max_items].contiguous()
        self.bank_contexts = torch.cat(contexts, dim=0)[:max_items].contiguous()

        # normalise for cosine similarity
        self.bank_embeds = F.normalize(self.bank_embeds, dim=1)

        print(f"Built retrieval bank with {self.bank_embeds.shape[0]} items")
    
    # retrive most similar context from bank
    @torch.no_grad()
    def query(self, query_context_latent):
        assert self.bank_embeds is not None, "Build the retrieval bank first."
        # compute pooled query embedding
        q = pooled_context_embedding(query_context_latent)
        q = F.normalize(q, dim=1)

        # cosine similarity using matrix multiply
        sims = torch.matmul(q.cpu(), self.bank_embeds.T)
        best_scores, best_idx = torch.max(sims, dim=1) # top 1 retrieval

        # fetch retrieved context
        retrieved = self.bank_contexts[best_idx].to(query_context_latent.device)

        # return similarity score
        sim_scores = best_scores.to(query_context_latent.device).unsqueeze(1)

        return retrieved, sim_scores, best_idx

## **Loss Functions**

In [ ]:
# compute diffusion training loss
def compute_noiseloss(
    pred_noise,
    true_noise,
    lossfn="masked_mse",
    delta=1.0,
    mask_latent=None,
    masked_weight=3.0,
    schedule=None,
    t=None,
    use_min_snr=True,
    min_snr_gamma=5.0,
):
    if lossfn == "mse":
        if use_min_snr:
            per_sample = per_sample_mse_loss(pred_noise, true_noise)
            weight = min_snr(schedule, t, gamma=min_snr_gamma)
            return (per_sample * weight).mean()
        else:
            return diffusion_noise_mse_loss(pred_noise, true_noise)

    elif lossfn == "masked_mse":
        if mask_latent is None:
            raise ValueError("mask_latent must be provided for masked_mse")

        if use_min_snr:
            per_sample = per_sample_masked_mse_loss(
                pred=pred_noise,
                target=true_noise,
                mask_latent=mask_latent,
                masked_weight=masked_weight,
            )
            weight = min_snr(schedule, t, gamma=min_snr_gamma)
            return (per_sample * weight).mean()
        else:
            return masked_diffusion_noise_mse_loss(
                pred_noise=pred_noise,
                true_noise=true_noise,
                mask_latent=mask_latent,
                masked_weight=masked_weight,
            )

    elif lossfn == "l1":
        return diffusion_noise_l1_loss(pred_noise, true_noise)

    elif lossfn == "huber":
        return diffusion_noise_huber_loss(pred_noise, true_noise, delta=delta)

    else:
        raise ValueError(f"Unsupported lossfn: {lossfn}")

# auxiliary latent reconstruction loss
def compute_latentloss(pred_latent, target_latent, lossfn="l1"):
    """
    after predicting noise, we can reconstruct an estimate of the clean latent z_0 hat
    this loss encourage z0_hat to stay close to the true clean latent z_clean
    """
    if lossfn == "l1":
        return latent_l1_loss(pred_latent, target_latent)
    elif lossfn == "l2":
        return latent_l2_loss(pred_latent, target_latent)
    else:
        raise ValueError(f"Unsupported latent loss: {lossfn}")

# creates a thin boundary band around the gap edges
def make_boundary_band(mask_latent, kernel_size=3):
    """ 
    inpainting bad joins often happen near gap boundaries. this band focuses a loss on the seam region
    """
    dilated = F.max_pool2d(mask_latent, kernel_size=kernel_size, stride=1, padding=kernel_size // 2)
    eroded = -F.max_pool2d(-mask_latent, kernel_size=kernel_size, stride=1, padding=kernel_size // 2)
    band = (dilated - eroded).clamp(0.0, 1.0)
    return band

def boundary_l1_loss(pred_latent, target_latent, mask_latent, kernel_size=3, eps=1e-8):
    band = make_boundary_band(mask_latent, kernel_size=kernel_size)
    diff = torch.abs(pred_latent - target_latent)
    band = band.expand_as(diff)
    return (diff * band).sum() / (band.sum() + eps)

## **Main Diffusion Wrapper**

In [ ]:
class DiffusionWrapper(nn.Module):
    """
    wraps diffusion unet, local context encoder, retreival endocder, and confidence gate
    """
    def __init__(
        self,
        latent_channels=8,
        base_channels=128,
        time_dim=256,
        cond_dim=256,
        context_base_channels=64,
    ):
        super().__init__()

        # encoder for local known context
        self.context_encoder = ContextEncoder(
            latent_channels=latent_channels,
            cond_dim=cond_dim,
            base_channels=context_base_channels,
        )
        # encoder for retrieved similar example
        self.retrieval_encoder = RetrievalEncoder(
            latent_channels=latent_channels,
            cond_dim=cond_dim,
            base_channels=context_base_channels,
        )
        
        # scalr gate to decide how much to trust retrieval
        self.confidence_gate = ConfidenceGate(hidden_dim=32)

        # main denoising unet
        self.unet = Diffusion_UNet(
            latent_channels=latent_channels,
            base_channels=base_channels,
            time_dim=time_dim,
            cond_dim=cond_dim,
        )

        self.latent_channels = latent_channels
        self.time_dim = time_dim
        self.cond_dim = cond_dim

    # build final conditioning tokens used by bottleneck crossattention
    def build_condition_tokens(self, z_context, m_latent, z_retrieved_context, sim_score, gap_ratio, t):
        """
        1. encode local known context into tokens
        2. encode retrieved context into tokens
        3. compute a scalar confidence gate
        4. scale retrieval tokens by that gate
        5. concatenate local + retrieved tokens
        """
        # local context tokens
        context_tokens = self.context_encoder(z_context, m_latent)

        # retrieval tokens
        retrieval_tokens = self.retrieval_encoder(z_retrieved_context)

        # normalise timestep into [0, 1]
        t_norm = t.float().unsqueeze(1) / float(max(t.max().item(), 1))

        # gate represenet how much retrieval should influence the model
        gate = self.confidence_gate(
            sim_score=sim_score,
            gap_ratio=gap_ratio,
            t_norm=t_norm,
        ) 
        gated_retrieval_tokens = retrieval_tokens * gate.unsqueeze(-1)

        # combine local and retrieval conditioning
        cond_tokens = torch.cat([context_tokens, gated_retrieval_tokens], dim=1)
        return cond_tokens, gate

    # delegates the modules into unet
    def forward(self, model_input, t, cond_tokens):
        return self.unet(model_input, t, cond_tokens)


In [ ]:
# self conditioning helper
@torch.no_grad()
def build_self_condition(
    model,
    z_t,
    z_masked,
    m_latent,
    cond_tokens,
    t,
    schedule,
    p_selfcond=0.5
):
    """
    during training:
      - sometimes run a no-grad first pass
      - generate detached z0_hat
      - feed it back as self-conditioning
    """
    if torch.rand(1).item() < p_selfcond:
        zero_selfcond = torch.zeros_like(z_t)

        # first no-grad pass
        model_input_prev = torch.cat([z_t, z_masked, m_latent, zero_selfcond], dim=1)
        eps_hat_prev = model(model_input_prev, t, cond_tokens)

        # convert predicted noise into estimated clean latent
        z0_selfcond = predict_x0(z_t, eps_hat_prev, t, schedule).detach()
    else:
        z0_selfcond = torch.zeros_like(z_t)

    return z0_selfcond

In [ ]:
# single training or eval step
def diffusion_step(
    vae,
    model,
    retrieval_bank,
    schedule,
    batch,
    device,
    noise_loss="masked_mse",
    latent_loss="l1",
    latent_loss_weight=0.05,
    boundary_loss_weight=0.05,
    boundary_kernel_size=3,
    delta=1.0,
    masked_weight=3.0,
    use_min_snr=True,
    min_snr_gamma=5.0,
    use_self_condition=True,
    p_selfcond=0.5,
):
    """
    1. loads a batch
    2. encodes clean and masked spectrograms with frozen VAE
    3. builds local context
    4. retrieves a similar example
    5. samples diffusion timestep and noise
    6. builds self-conditioning
    7. predicts noise with the model
    8. computes losses
    """
    x = batch["x"].to(device, non_blocking=True)
    y = batch["y"].to(device, non_blocking=True)
    m = batch["mask"].to(device, non_blocking=True)

    x, _ = padding(x, multiple=16)
    y, _ = padding(y, multiple=16)
    m, _ = padding(m, multiple=16)

    # encode clean and masked spectrograms using frozen VAE
    with torch.no_grad():
        z_clean = encode2latentmean(vae, y)
        z_masked = encode2latentmean(vae, x)

    # resize mask to latent resolution
    m_latent = F.interpolate(m, size=z_clean.shape[-2:], mode="nearest")

    # local known context
    z_context = z_masked * (1.0 - m_latent)

    # retrieve top-1 similar context from retrieval bank
    with torch.no_grad():
        z_retrieved_context, sim_score, _ = retrieval_bank.query(z_context)

    # gap ratio is a simple scalar telling how large the missing region is
    gap_ratio = compute_gap_ratio(m_latent).view(z_clean.shape[0], 1)

    # diffusion timestep + noise
    t = torch.randint(
        low=0,
        high=schedule.betas.shape[0],
        size=(z_clean.shape[0],),
        device=device,
    ).long()

    noise = torch.randn_like(z_clean) # sample gaussian noise
    z_t = q_sample(z_clean, t, noise, schedule) # create noise latent z_t from clean latent z_clean

    # condition tokens
    cond_tokens, gate = model.build_condition_tokens(
        z_context=z_context,
        m_latent=m_latent,
        z_retrieved_context=z_retrieved_context,
        sim_score=sim_score,
        gap_ratio=gap_ratio,
        t=t,
    )

    # self-conditioning
    if use_self_condition:
        z0_selfcond = build_self_condition(
            model=model,
            z_t=z_t,
            z_masked=z_masked,
            m_latent=m_latent,
            cond_tokens=cond_tokens,
            t=t,
            schedule=schedule,
            p_selfcond=p_selfcond,
        )
    else:
        z0_selfcond = torch.zeros_like(z_clean)

    # final model input
    model_input = torch.cat([z_t, z_masked, m_latent, z0_selfcond], dim=1)

    # predict noise
    eps_hat = model(model_input, t, cond_tokens)

    # main diffusion noise loss
    noise_loss_value = compute_noiseloss(
        pred_noise=eps_hat,
        true_noise=noise,
        lossfn=noise_loss,
        delta=delta,
        mask_latent=m_latent,
        masked_weight=masked_weight,
        schedule=schedule,
        t=t,
        use_min_snr=use_min_snr,
        min_snr_gamma=min_snr_gamma,
    )

    # estimate clean latent z0_hat from predicted noise
    z0_hat = predict_x0(z_t, eps_hat, t, schedule)

    # auxiliary latent reconstruction loss
    latent_loss_value = compute_latentloss(
        pred_latent=z0_hat,
        target_latent=z_clean,
        lossfn=latent_loss,
    )
    # boundary loss in latent space
    boundary_loss_value = boundary_l1_loss(
        pred_latent=z0_hat,
        target_latent=z_clean,
        mask_latent=m_latent,
        kernel_size=boundary_kernel_size,
    )
    # total loss
    total_loss = (
        noise_loss_value
        + latent_loss_weight * latent_loss_value
        + boundary_loss_weight * boundary_loss_value
    )

    return {
        "loss": total_loss,
        "noise_loss": noise_loss_value.detach(),
        "latent_loss": latent_loss_value.detach(),
        "boundary_loss": boundary_loss_value.detach(),
        "gate_mean": gate.mean().detach(),
    }

In [ ]:
# one training epoch
def trainDiffusion(
    vae,
    model,
    retrieval_bank,
    schedule,
    dataloader,
    optimiser,
    device,
    noise_loss="masked_mse",
    latent_loss="l1",
    latent_loss_weight=0.05,
    boundary_loss_weight=0.05,
    boundary_kernel_size=3,
    delta=1.0,
    masked_weight=3.0,
    use_min_snr=True,
    min_snr_gamma=5.0,
    use_self_condition=True,
    p_selfcond=0.5,
    grad_clip=1.0,
    use_amp=True,
):
    model.train()

    running = {
        "loss": 0.0,
        "noise_loss": 0.0,
        "latent_loss": 0.0,
        "boundary_loss": 0.0,
        "gate_mean": 0.0,
    }
    n_batches = 0

    use_amp = use_amp and ("cuda" in str(device))
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    for batch in tqdm(dataloader, desc="Train Diffusion", leave=False):
        optimiser.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda", enabled=use_amp):
            out = diffusion_step(
                vae=vae,
                model=model,
                retrieval_bank=retrieval_bank,
                schedule=schedule,
                batch=batch,
                device=device,
                noise_loss=noise_loss,
                latent_loss=latent_loss,
                latent_loss_weight=latent_loss_weight,
                boundary_loss_weight=boundary_loss_weight,
                boundary_kernel_size=boundary_kernel_size,
                delta=delta,
                masked_weight=masked_weight,
                use_min_snr=use_min_snr,
                min_snr_gamma=min_snr_gamma,
                use_self_condition=use_self_condition,
                p_selfcond=p_selfcond,
            )
        # backward pass with gradient scaling
        scaler.scale(out["loss"]).backward()

        # clip gradients for stability
        if grad_clip is not None:
            scaler.unscale_(optimiser)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip)

        scaler.step(optimiser)
        scaler.update()
        
        # accumulate running metrics
        for k in running:
            running[k] += out[k].item()

        n_batches += 1

    return {k: v / max(n_batches, 1) for k, v in running.items()}


In [ ]:
# one evaluation epoch
@torch.no_grad()
def evalDiffusion(
    vae,
    model,
    retrieval_bank,
    schedule,
    dataloader,
    device,
    noise_loss="masked_mse",
    latent_loss="l1",
    latent_loss_weight=0.05,
    boundary_loss_weight=0.05,
    boundary_kernel_size=3,
    delta=1.0,
    masked_weight=3.0,
    use_min_snr=True,
    min_snr_gamma=5.0,
    use_self_condition=True,
    p_selfcond=0.5,
    use_amp=True,
):
    model.eval()

    running = {
        "loss": 0.0,
        "noise_loss": 0.0,
        "latent_loss": 0.0,
        "boundary_loss": 0.0,
        "gate_mean": 0.0,
    }
    n_batches = 0

    use_amp = use_amp and ("cuda" in str(device))

    for batch in tqdm(dataloader, desc="Eval Diffusion", leave=False):
        with torch.amp.autocast("cuda", enabled=use_amp):
            out = diffusion_step(
                vae=vae,
                model=model,
                retrieval_bank=retrieval_bank,
                schedule=schedule,
                batch=batch,
                device=device,
                noise_loss=noise_loss,
                latent_loss=latent_loss,
                latent_loss_weight=latent_loss_weight,
                boundary_loss_weight=boundary_loss_weight,
                boundary_kernel_size=boundary_kernel_size,
                delta=delta,
                masked_weight=masked_weight,
                use_min_snr=use_min_snr,
                min_snr_gamma=min_snr_gamma,
                use_self_condition=use_self_condition,
                p_selfcond=p_selfcond,
            )

        for k in running:
            running[k] += out[k].item()

        n_batches += 1

    return {k: v / max(n_batches, 1) for k, v in running.items()}

In [ ]:
# fit model
def fitDiffusion(
    vae,
    model,
    retrieval_bank,
    schedule,
    train_loader,
    val_loader,
    optimiser,
    device,
    n_epochs,
    checkpoint_dir,
    history_path,

    noise_loss="masked_mse",
    latent_loss="l1",
    latent_loss_weight=0.05,
    boundary_loss_weight=0.05,
    boundary_kernel_size=3,
    delta=1.0,
    masked_weight=3.0,

    use_min_snr=True,
    min_snr_gamma=5.0,

    use_self_condition=True,
    p_selfcond=0.5,

    grad_clip=1.0,
    use_amp=True,

    monitor="val_loss",
    mode="min",
    patience=10,
    min_delta=1e-4,
    save_best_after_epoch=1,

    use_scheduler=True,
    scheduler_type="plateau",
    scheduler_factor=0.5,
    scheduler_patience=3,
    scheduler_min_lr=1e-7,

    resume_checkpoint_path=None,
    load_history=True,
):
    checkpoint_dir = Path(checkpoint_dir)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)

    history_path = Path(history_path)
    history_path.parent.mkdir(parents=True, exist_ok=True)

    # checkpoint manager handles saving best and last models
    manager = ModelCheckpoint(
        checkpoint_dir=checkpoint_dir,
        monitor=monitor,
        mode=mode,
        patience=patience,
        min_delta=min_delta,
        save_best_after_epoch=save_best_after_epoch,
        verbose=True,
    )
    # history storage
    history = {
        "epoch": [],
        "lr": [],
        "train_loss": [],
        "train_noise_loss": [],
        "train_latent_loss": [],
        "train_boundary_loss": [],
        "train_gate_mean": [],
        "val_loss": [],
        "val_noise_loss": [],
        "val_latent_loss": [],
        "val_boundary_loss": [],
        "val_gate_mean": [],
    }

    start_epoch = 1

    # resume traininng
    if resume_checkpoint_path is not None:
        resume_checkpoint_path = Path(resume_checkpoint_path)
        if resume_checkpoint_path.exists():
            ckpt = torch.load(resume_checkpoint_path, map_location=device)
            model.load_state_dict(ckpt["model_state_dict"])

            if "optimiser_state_dict" in ckpt:
                optimiser.load_state_dict(ckpt["optimiser_state_dict"])

            start_epoch = ckpt.get("epoch", 0) + 1
            manager.best_score = ckpt.get("best_score", manager.best_score)
            manager.best_epoch = ckpt.get("best_epoch", manager.best_epoch)

            print(f"Resumed from: {resume_checkpoint_path}")
            print(f"Continuing from epoch: {start_epoch}")

    # load old csv history 
    if load_history and history_path.exists():
        old_history_df = pd.read_csv(history_path)
        missing_cols = [k for k in history.keys() if k not in old_history_df.columns]
        if len(missing_cols) == 0:
            history = {col: old_history_df[col].tolist() for col in old_history_df.columns}
            print(f"Loaded existing history from: {history_path}")

    # build lr scheduler
    scheduler = None
    if use_scheduler:
        if scheduler_type == "plateau":
            scheduler = ReduceLROnPlateau(
                optimiser,
                mode=mode,
                factor=scheduler_factor,
                patience=scheduler_patience,
                min_lr=scheduler_min_lr,
            )
        elif scheduler_type == "cosine":
            scheduler = CosineAnnealingLR(
                optimiser,
                T_max=n_epochs,
                eta_min=scheduler_min_lr,
            )
        else:
            raise ValueError(f"Unsupported scheduler_type: {scheduler_type}")

    # main epoch loop
    for epoch in tqdm(range(start_epoch, n_epochs + 1), desc="Training Diffusion"):
        # train
        train_metrics = trainDiffusion(
            vae=vae,
            model=model,
            retrieval_bank=retrieval_bank,
            schedule=schedule,
            dataloader=train_loader,
            optimiser=optimiser,
            device=device,
            noise_loss=noise_loss,
            latent_loss=latent_loss,
            latent_loss_weight=latent_loss_weight,
            boundary_loss_weight=boundary_loss_weight,
            boundary_kernel_size=boundary_kernel_size,
            delta=delta,
            masked_weight=masked_weight,
            use_min_snr=use_min_snr,
            min_snr_gamma=min_snr_gamma,
            use_self_condition=use_self_condition,
            p_selfcond=p_selfcond,
            grad_clip=grad_clip,
            use_amp=use_amp,
        )

        # val
        val_metrics = evalDiffusion(
            vae=vae,
            model=model,
            retrieval_bank=retrieval_bank,
            schedule=schedule,
            dataloader=val_loader,
            device=device,
            noise_loss=noise_loss,
            latent_loss=latent_loss,
            latent_loss_weight=latent_loss_weight,
            boundary_loss_weight=boundary_loss_weight,
            boundary_kernel_size=boundary_kernel_size,
            delta=delta,
            masked_weight=masked_weight,
            use_min_snr=use_min_snr,
            min_snr_gamma=min_snr_gamma,
            use_self_condition=use_self_condition,
            p_selfcond=p_selfcond,
            use_amp=use_amp,
        )

        # update scheduler
        if scheduler is not None:
            if scheduler_type == "plateau":
                scheduler.step(val_metrics["loss"])
            else:
                scheduler.step()

        current_lr = get_lr(optimiser)

        # report epoch metrics
        epoch_record = {
            "epoch": epoch,
            "lr": current_lr,
            "train_loss": train_metrics["loss"],
            "train_noise_loss": train_metrics["noise_loss"],
            "train_latent_loss": train_metrics["latent_loss"],
            "train_boundary_loss": train_metrics["boundary_loss"],
            "train_gate_mean": train_metrics["gate_mean"],
            "val_loss": val_metrics["loss"],
            "val_noise_loss": val_metrics["noise_loss"],
            "val_latent_loss": val_metrics["latent_loss"],
            "val_boundary_loss": val_metrics["boundary_loss"],
            "val_gate_mean": val_metrics["gate_mean"],
        }
        # append into history
        for key in history:
            history[key].append(epoch_record[key])

        pd.DataFrame(history).to_csv(history_path, index=False)

        # print summary
        print(f"Epoch {epoch:02d}")
        print(f"Learning Rate:       {current_lr:.8e}")
        print(f"Train Loss:          {train_metrics['loss']:.6f}")
        print(f"Val Loss:            {val_metrics['loss']:.6f}")
        print(f"Train Noise Loss:    {train_metrics['noise_loss']:.6f}")
        print(f"Val Noise Loss:      {val_metrics['noise_loss']:.6f}")
        print(f"Train Latent Loss:   {train_metrics['latent_loss']:.6f}")
        print(f"Val Latent Loss:     {val_metrics['latent_loss']:.6f}")
        print(f"Train Boundary Loss: {train_metrics['boundary_loss']:.6f}")
        print(f"Val Boundary Loss:   {val_metrics['boundary_loss']:.6f}")
        print(f"Train Gate Mean:     {train_metrics['gate_mean']:.4f}")
        print(f"Val Gate Mean:       {val_metrics['gate_mean']:.4f}")
        print("-" * 60)

        # checkpointing/early stopping
        manager.step(
            epoch=epoch,
            metrics=epoch_record,
            model=model,
            optimiser=optimiser,
        )

        if manager.should_stop:
            print(f"Early stopping triggered at epoch {epoch}")
            break

    return {
        "history": history,
        "best_score": manager.best_score,
        "best_epoch": manager.best_epoch,
        "checkpoint_dir": checkpoint_dir,
    }

In [ ]:
# build retrieval bank
retrieval_bank = RetrievalBank(device=device)
retrieval_bank.build(
    vae=vae,
    dataloader=train_loader,
    device=device,
    max_items=4000,
)

In [ ]:
# initialise diffusion
diffusion_model = DiffusionWrapper(
    latent_channels=8,
    base_channels=128,
    time_dim=256,
    cond_dim=256,
    context_base_channels=64
).to(device)

In [ ]:
# build optimiser
diffusion_optimiser=AdamW(diffusion_model.parameters(), lr=1e-4, weight_decay=1e-4)
schedule = cosine_schedule(num_steps=1000, device=device)

checkpoint_dir = root_dir / "diffusion_improv2_" / "checkpoints"
history_dir = root_dir / "diffusion_improv2_" / "history"

In [ ]:
diffusion_results = fitDiffusion(
    vae=vae,
    model=diffusion_model,
    retrieval_bank=retrieval_bank,
    schedule=schedule,
    train_loader=train_loader,
    val_loader=val_loader,
    optimiser=diffusion_optimiser,
    device=device,
    n_epochs=150,
    checkpoint_dir=checkpoint_dir,
    history_path=history_dir / "history.csv",

    noise_loss="masked_mse",
    latent_loss="l1",
    latent_loss_weight=0.05,
    boundary_loss_weight=0.05,
    boundary_kernel_size=3,
    delta=1.0,
    masked_weight=3.0,

    use_min_snr=True,
    min_snr_gamma=5.0,

    use_self_condition=True,
    p_selfcond=0.5,

    grad_clip=1.0,
    use_amp=True,

    monitor="val_loss",
    mode="min",
    patience=10,
    min_delta=1e-4,
    save_best_after_epoch=1,

    use_scheduler=True,
    scheduler_type="plateau",
    scheduler_factor=0.5,
    scheduler_patience=3,
    scheduler_min_lr=1e-7,

    resume_checkpoint_path=checkpoint_dir / "last_model.pt",
    load_history=True,
)

#### **Inferencing/Sampling Helper Functions**

In [ ]:
# one reverse diffusion step p(z_{t-1} | z_t, conditioning)
@torch.no_grad()
def p_sample(
    model,
    z_t,
    z_masked,
    mask_latent,
    z0_selfcond,
    cond_tokens,
    t,
    schedule,
    add_noise=True,
):
    # build the input expected by the diffusion U-Net.
    # the model sees:
    # - current noisy latent z_t
    # - latent of masked spectrogram z_masked
    # - latent binary mask
    # - previous clean estimate z0_selfcond
    model_input = torch.cat([z_t, z_masked, mask_latent, z0_selfcond], dim=1)

    # predict the diffusion noise eps_hat from the current noisy latent and the conditioning information
    eps_hat = model(model_input, t, cond_tokens)

    # extract timestep-specific schedule values and reshape them so they broadcast correctly over latent tensors
    beta_t = extract(schedule.betas, t, z_t.shape)
    sqrt_one_minus_alpha_bar_t = extract(schedule.sqrt_one_minus_alpha_bars, t, z_t.shape)
    sqrt_recip_alpha_t = extract(schedule.sqrt_recip_alphas, t, z_t.shape)
    posterior_var_t = extract(schedule.posterior_variance, t, z_t.shape)

    # DDPM reverseprocess mean
    model_mean = sqrt_recip_alpha_t * (
        z_t - (beta_t / sqrt_one_minus_alpha_bar_t) * eps_hat
    )

    # at t=0, no extra nosie should be added so nonzero_mask is used
    if add_noise:
        noise = torch.randn_like(z_t)
        nonzero_mask = (t != 0).float().view(z_t.shape[0], 1, 1, 1)
        z_prev = model_mean + nonzero_mask * torch.sqrt(posterior_var_t) * noise
    else:
        # deterministic reverse step
        z_prev = model_mean

    # recovers model's estimate of the original clean latene z_0
    z0_hat = predict_x0(z_t, eps_hat, t, schedule)
    return z_prev, z0_hat, eps_hat

# full latent space pipeline for inpaining
@torch.no_grad()
def inferencing_latent(
    vae,
    model,
    retrieval_bank,
    schedule,
    x_masked,
    mask,
    device,
    num_steps=None,
    add_noise=True,
    return_all_steps=False,
):
    """
    - encode the masked spectrogram into latent space
    - build local context and retrieval conditioning
    - start from Gaussian latent noise
    - repeatedly denoise using the trained model
    - decode the final latent back into a spectrogram
    """
    vae.eval()
    model.eval()

    if num_steps is None:
        num_steps = len(schedule.betas)

    x_masked, pad_info = padding(x_masked, multiple=16)
    mask, _ = padding(mask, multiple=16)

    z_masked = encode2latentmean(vae, x_masked)
    mask_latent = F.interpolate(mask, size=z_masked.shape[-2:], mode="nearest")
    z_context = z_masked * (1.0 - mask_latent)

    # retrieval
    z_retrieved_context, sim_score, _ = retrieval_bank.query(z_context)
    gap_ratio = compute_gap_ratio(mask_latent).view(z_masked.shape[0], 1)

    # initial latent sample
    z_t = torch.randn_like(z_masked)

    all_steps = [z_t.detach().cpu()] if return_all_steps else None
    z0_selfcond = torch.zeros_like(z_masked)

    timesteps = list(range(len(schedule.betas) - 1, -1, -1))
    if num_steps < len(timesteps):
        # simple stride-based subsampling
        idxs = np.linspace(0, len(timesteps) - 1, num_steps).round().astype(int)
        timesteps = [timesteps[i] for i in sorted(set(idxs), reverse=True)]

    for t_scalar in tqdm(timesteps, desc="Sampling", leave=False):
        t = torch.full((z_t.shape[0],), t_scalar, device=device, dtype=torch.long)

        cond_tokens, gate = model.build_condition_tokens(
            z_context=z_context,
            m_latent=mask_latent,
            z_retrieved_context=z_retrieved_context,
            sim_score=sim_score,
            gap_ratio=gap_ratio,
            t=t,
        )

        z_t, z0_hat, eps_hat = p_sample(
            model=model,
            z_t=z_t,
            z_masked=z_masked,
            mask_latent=mask_latent,
            z0_selfcond=z0_selfcond,
            cond_tokens=cond_tokens,
            t=t,
            schedule=schedule,
            add_noise=add_noise,
        )

        z0_selfcond = z0_hat.detach()

        if return_all_steps:
            all_steps.append(z_t.detach().cpu())

    x_hat = decode_from_latent(vae, z_t)
    x_hat = unpadding(x_hat, pad_info)

    out = {
        "recon_spec": x_hat,
        "final_latent": z_t,
    }

    if return_all_steps:
        out["all_steps"] = all_steps

    return out

# run full latent space inference on one batch, then compute reconstruction metrics
@torch.no_grad()
def reconstruct_batch(
    vae,
    model,
    retrieval_bank,
    schedule,
    batch,
    device,
    num_steps=None,
    add_noise=True,
):
    x = batch["x"].to(device)
    y = batch["y"].to(device)
    m = batch["mask"].to(device)

    out = inferencing_latent(
        vae=vae,
        model=model,
        retrieval_bank=retrieval_bank,
        schedule=schedule,
        x_masked=x,
        mask=m,
        device=device,
        num_steps=num_steps,
        add_noise=add_noise,
        return_all_steps=False,
    )

    recon = out["recon_spec"]

    return {
        "recon": recon,
        "y": y,
        "mask": m,
        "gap_mae": masked_mae(recon, y, m).item(),
        "gap_rmse": masked_rmse(recon, y, m).item(),
        "full_mae": full_mae(recon, y).item(),
        "full_rmse": full_rmse(recon, y).item(),
        "psnr": psnr(recon, y).item(),
    }